# The Meta Behind Siege: Operator Balance, Tactical Roles & Hidden Patterns in Rainbow Six

Most shooters reward the player with the fastest crosshair. Siege punishes that player.

Rainbow Six Siege is a game where a 150-IQ smoke setup can beat a 200-IQ flick, where a single piece of utility – a Mute jammer, a Thatcher EMP, a Mira mirror – can swing a round that aim alone could never win. The drone phase often decides the fight before a single shot is fired. That's the whole identity of the game: **utility first, gunfights second.**

That makes balance a nightmare. Ubisoft isn't just tuning damage numbers – they're tuning *information*, *map control*, *denial*, and *synergy*. Buff one gadget and four operators disappear from ranked. Nerf one acog and an entire site rework becomes obsolete.

This notebook is an honest, player-first look at the operator and weapon data behind the meta:

- Who's actually overloaded with utility?
- Which operators are *one-trick* picks the casual eye misses?
- What does the weapon archetype map look like once you strip away pick rate?
- Can we cluster the roster into the same archetypes pros call them in scrims – fraggers, anchors, roamers, intel, support?
- And the fun one: **if you main Ash, who should you try next?**

Dataset: [Rainbow Six Siege Encyclopedia](https://www.kaggle.com/) – operators, weapons, gadgets, charms, maps, seasons.


---
## 1. Data Loading & Overview

Six CSVs, nothing huge. Standard rule: load, look, trust nothing until the dtypes line up.


In [ ]:
import os, glob, ast, warnings, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)

# Siege-flavoured palette: orange (attacker), blue (defender), tac-gray accents
ATK, DEF = "#ff8c1a", "#3aa0ff"
ACCENT, GOLD, MINT = "#e23d3d", "#ffd166", "#22e3a3"
PALETTE = [ATK, DEF, GOLD, MINT, ACCENT, "#a78bfa"]

sns.set_theme(style="darkgrid", rc={
    "figure.facecolor": "#0b0d12",
    "axes.facecolor":   "#11151d",
    "axes.edgecolor":   "#222a36",
    "axes.labelcolor":  "#d6d8df",
    "xtick.color":      "#a8adb8",
    "ytick.color":      "#a8adb8",
    "text.color":       "#e6e8ee",
    "axes.titleweight": "bold",
    "axes.titlesize":   13,
    "grid.color":       "#1c2230",
})
PLOTLY_TEMPLATE = "plotly_dark"

# ---- Data source resolution ----------------------------------------------
# On Kaggle the dataset is auto-mounted under /kaggle/input.
# Locally, we either use the ./data folder or fall back to kagglehub.
KAGGLE_SLUG = "ektarr/rainbow-six-siege-encyclopedia"
CANDIDATE_DIRS = [
    "/kaggle/input/rainbow-six-siege-encyclopedia",
    "/kaggle/input/rainbow-six-siege-encyclopedia/data",
    "data",
    ".",
]

def _has_csvs(p): return os.path.isdir(p) and bool(glob.glob(os.path.join(p, "*.csv")))

DATA_DIR = next((p for p in CANDIDATE_DIRS if _has_csvs(p)), None)

if DATA_DIR is None:
    # Pull from Kaggle via kagglehub (works locally if you have kaggle creds set up)
    try:
        import kagglehub
        DATA_DIR = kagglehub.dataset_download(KAGGLE_SLUG)
        # kagglehub may nest the csvs under a /data subfolder
        if not _has_csvs(DATA_DIR) and _has_csvs(os.path.join(DATA_DIR, "data")):
            DATA_DIR = os.path.join(DATA_DIR, "data")
    except Exception as e:
        raise SystemExit(f"Could not locate dataset. Install kagglehub or mount the data folder. ({e})")

print("Data dir:", DATA_DIR)
print("Files:", [os.path.basename(p) for p in sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))])


In [ ]:
def load(name):
    return pd.read_csv(os.path.join(DATA_DIR, name))

operators   = load("operators.csv")
weapons     = load("weapons.csv")
maps_df     = load("maps.csv")
seasons     = load("seasons.csv")
attachments = load("attachments.csv")
charms      = load("charms.csv")

print(f"Operators : {operators.shape}")
print(f"Weapons   : {weapons.shape}")
print(f"Maps      : {maps_df.shape}")
print(f"Seasons   : {seasons.shape}")
print(f"Attachments: {attachments.shape}")
print(f"Charms    : {charms.shape}")
operators.head(3)


In [ ]:
# Clean operators: parse roles, normalize unit casing, derive Year from season code
def parse_roles(s):
    try:
        out = ast.literal_eval(s) if isinstance(s, str) else []
        return [r.strip().title() for r in out]
    except Exception:
        return []

ops = operators.copy()
ops["roles_list"] = ops["roles"].apply(parse_roles)
ops["n_roles"]    = ops["roles_list"].apply(len)
ops["unit"]       = ops["unit"].fillna("Unknown").str.upper().str.strip()
ops["country_code"] = ops["country_code"].fillna("??")
ops["side"]       = ops["side"].str.lower()

# Year/season number from "YxxSx"
def parse_year(code):
    m = re.match(r"Y(\d+)S(\d+)", str(code))
    return (int(m.group(1)), int(m.group(2))) if m else (0, 0)
ops[["year_intro","season_num"]] = ops["season_introduced"].apply(lambda c: pd.Series(parse_year(c)))

# Quick data quality
miss = pd.concat([
    operators.isna().mean().mul(100).rename("operators_%miss"),
    weapons.isna().mean().mul(100).rename("weapons_%miss"),
], axis=1).fillna(0).round(1)
miss = miss[(miss.sum(axis=1) > 0)]
print("Missing-value snapshot:")
print(miss)
print(f"\nUnique roles tagged across roster: {len({r for lst in ops['roles_list'] for r in lst})}")


**Honest read of the data.** It's small but dense – 78 operators, ~110 guns, and clean health/speed/role tags. The `roles` column is the gold here: it's pre-tagged by Ubisoft with the tactical identity they *think* each operator has. Whether the live meta agrees with them is a different question, and one we'll poke at later.

---
## 3. Operator Landscape Analysis

The first thing any Siege player checks: who's on the roster, what side, how much HP, how fast. That's the chassis – everything else (gadget, gun) sits on top.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# Attacker vs Defender
side_counts = ops["side"].value_counts()
axes[0].bar(side_counts.index.str.title(), side_counts.values,
            color=[ATK if s == "attacker" else DEF for s in side_counts.index])
for i, v in enumerate(side_counts.values):
    axes[0].text(i, v + 0.3, str(v), ha="center", color="white", fontweight="bold")
axes[0].set_title("Attackers vs Defenders")
axes[0].set_ylabel("# operators")

# Speed/HP chassis grid
chassis = ops.groupby(["speed","health"]).size().unstack(fill_value=0)
sns.heatmap(chassis, annot=True, fmt="d", cmap="rocket_r", cbar=False,
            linewidths=0.6, linecolor="#0b0d12", ax=axes[1])
axes[1].set_title("Chassis grid — speed × health")
axes[1].set_xlabel("Health (armor)"); axes[1].set_ylabel("Speed")

# Roster growth by year
ops_year = ops.groupby("year_intro").size().rename("new_ops")
ops_year = ops_year[ops_year.index > 0]
axes[2].fill_between(ops_year.index, ops_year.values, color=ATK, alpha=0.35)
axes[2].plot(ops_year.index, ops_year.values, color=ATK, marker="o", lw=2)
axes[2].set_title("New operators per Year")
axes[2].set_xlabel("Year (Y1 = 2016)"); axes[2].set_ylabel("# released")

plt.tight_layout(); plt.show()


**Meta read.** The roster is dead-even attacker/defender — Ubisoft has held that line religiously since launch. The chassis grid is the more interesting tell: **2-speed/2-health is the default**, and the corners (3/1 fragger or 1/3 anchor) stay scarce. That's a balance decision — 3-speed operators warp pacing, and 3-armor anchors get oppressive on bomb sites. Yearly releases dropped after the early gold rush, which tracks with Ubisoft's pivot from "ten new operators a year" to "fewer ops, more reworks."


In [ ]:
# Role frequency, split by side
role_long = ops[["name","side","roles_list"]].explode("roles_list").dropna()
role_long = role_long.rename(columns={"roles_list":"role"})
role_side = (role_long.groupby(["role","side"]).size()
                       .unstack(fill_value=0)
                       .assign(total=lambda d: d.sum(axis=1))
                       .sort_values("total", ascending=True))
role_side_plot = role_side.tail(18).drop(columns="total")

fig, ax = plt.subplots(figsize=(9, 6.5))
role_side_plot.plot(kind="barh", stacked=True, color=[ATK, DEF],
                    edgecolor="#0b0d12", ax=ax, width=0.78)
ax.set_title("Tactical roles across the roster (Ubisoft tags)")
ax.set_xlabel("# operators carrying this role tag")
ax.set_ylabel("")
ax.legend(title="Side", labels=["Attacker","Defender"])
plt.tight_layout(); plt.show()


In [ ]:
# Nationality + organization (treemap)
nat = ops["country_code"].value_counts().rename_axis("country").reset_index(name="n")
unit = (ops["unit"].replace({"NIGHTHAVEN":"Nighthaven"})
                   .str.title().value_counts()
                   .rename_axis("unit").reset_index(name="n"))

fig = px.treemap(unit, path=["unit"], values="n", color="n",
                 color_continuous_scale="Oranges", template=PLOTLY_TEMPLATE,
                 title="Counter-terror units represented on the Rainbow roster")
fig.update_layout(height=460, coloraxis_showscale=False, margin=dict(t=60,l=10,r=10,b=10))
fig.show()

fig = px.bar(nat.head(15).sort_values("n"), x="n", y="country", orientation="h",
             template=PLOTLY_TEMPLATE, color="n", color_continuous_scale="Blues",
             title="Operator nationalities — top 15")
fig.update_layout(height=440, coloraxis_showscale=False, yaxis_title="", xaxis_title="# operators")
fig.show()


**Meta read.** The big legacy units — GIGN, SAS, Spetsnaz, GSG-9 — anchor the roster, but **Nighthaven** has quietly turned into the new "anything goes" faction Ubisoft drops modern operators into when they don't want to commit to a real-world unit (Kali, Aruni, Osa, Ace…). It's basically the merc bin for operators whose kits don't fit a national CT framework.

---
## 4. Weapon & Loadout Intelligence

Siege guns lie to your eyes. Damage numbers are *per-bullet* and don't tell you anything until you fold in fire rate, ammo, and how forgiving the recoil is. We'll build a couple of fair efficiency metrics from what's in the data and let the meta picks float to the top.


In [ ]:
w = weapons.copy()
# Normalize a typo or two ("Machine Pustol" -> "Machine Pistol")
w["type"] = w["type"].replace({"Machine Pustol": "Machine Pistol"})
w["operators_list"] = w["operators"].fillna("").apply(
    lambda s: [x.strip() for x in s.split(";") if x.strip()])
w["n_users"] = w["operators_list"].apply(len)

# Derived metrics. These are simple but honest:
#   DPS proxy            = damage * fire rate / 60
#   ammo_capacity_score  = mag size / 30 (1.0 ~ "standard" AR mag)
#   ease_score           = inverse difficulty (1 hardest, 5 easiest in this data)
w["dps_proxy"] = w["stats_damage"] * w["stats_firerate"] / 60.0
w["ammo_score"] = w["stats_ammo"] / 30.0
# difficulty in this dataset: lower seems harder; flip to "ease"
w["ease_score"] = w["stats_difficulty"]

# Efficiency score: blend of dps, ammo, ease — z-scored then averaged
def z(s): return (s - s.mean()) / s.std(ddof=0)
w["efficiency"] = (z(w["dps_proxy"]) + 0.5*z(w["ammo_score"]) + 0.4*z(w["ease_score"])) / 1.9

type_counts = w["type"].value_counts()
print("Weapon classes on the roster:")
print(type_counts.to_string())


In [ ]:
primary_classes = ["Assault Rifle","Submachine Gun","Light Machine Gun","Marksman Rifle",
                   "Shotgun","Slug Shotgun","Sniper Rifle"]
wp = w[w["type"].isin(primary_classes)].copy()

fig = px.scatter(wp, x="stats_damage", y="stats_firerate", size="stats_ammo",
                 color="type", hover_name="name",
                 size_max=22, template=PLOTLY_TEMPLATE,
                 color_discrete_sequence=px.colors.qualitative.Vivid,
                 title="The damage/fire-rate map of primary weapons")
fig.update_layout(height=520, xaxis_title="Damage per bullet",
                  yaxis_title="Fire rate (RPM)")
fig.show()


In [ ]:
top_eff = (wp.sort_values("efficiency", ascending=False)
             .head(15)[["name","type","stats_damage","stats_firerate","stats_ammo","dps_proxy","efficiency","n_users"]]
             .round(2)
             .reset_index(drop=True))
top_eff


In [ ]:
# Class-level profile — mean stats per weapon class
class_profile = (wp.groupby("type")
                   .agg(damage=("stats_damage","mean"),
                        firerate=("stats_firerate","mean"),
                        ammo=("stats_ammo","mean"),
                        difficulty=("stats_difficulty","mean"),
                        dps=("dps_proxy","mean"),
                        count=("name","size"))
                   .round(1)
                   .sort_values("dps", ascending=False))
class_profile


In [ ]:
# Radar — average profile per primary class, scaled 0..1
radar_classes = ["Assault Rifle","Submachine Gun","Light Machine Gun","Marksman Rifle","Shotgun"]
radar_df = class_profile.loc[[c for c in radar_classes if c in class_profile.index],
                              ["damage","firerate","ammo","difficulty","dps"]].copy()
radar_norm = (radar_df - radar_df.min()) / (radar_df.max() - radar_df.min() + 1e-9)
radar_norm = radar_norm.round(2)

fig = go.Figure()
colors = px.colors.qualitative.Vivid
for i, cls in enumerate(radar_norm.index):
    vals = radar_norm.loc[cls].tolist()
    fig.add_trace(go.Scatterpolar(
        r=vals + [vals[0]],
        theta=list(radar_norm.columns) + [radar_norm.columns[0]],
        fill="toself", name=cls,
        line=dict(color=colors[i % len(colors)], width=2),
        opacity=0.55,
    ))
fig.update_layout(template=PLOTLY_TEMPLATE, height=520,
                  title="Class fingerprints — normalized stat profiles",
                  polar=dict(radialaxis=dict(visible=True, range=[0,1])))
fig.show()


In [ ]:
# Loadout flexibility per operator: how many distinct primary classes they can pick
op_primary = (w[w["type"].isin(primary_classes)][["name","type","operators_list"]]
                .explode("operators_list")
                .dropna()
                .rename(columns={"operators_list":"operator","name":"weapon"}))
flex = (op_primary.groupby("operator")
                  .agg(primary_options=("weapon","nunique"),
                       primary_classes=("type","nunique"))
                  .sort_values("primary_options", ascending=False))
flex_top = flex.head(12)
flex_bot = flex[flex["primary_options"] > 0].tail(12)
print("Most flexible loadouts:");      print(flex_top)
print("\nMost locked-in loadouts:");   print(flex_bot)


**Meta read.** The LMG and Marksman classes look strong by raw DPS, but that's a stats illusion — recoil and ADS time are what get them benched in real games. The interesting tell is **loadout flexibility**: operators with *one* primary class (Glaz with only his sniper, shotgun-only defenders) carry an enormous tactical opportunity cost. They demand the team build *around* them, which is exactly why those operators come and go from the meta on every patch.

---
## 5. Tactical Role Analysis

Ubisoft's role tags are a good starting point but they're inconsistent — "Intel" and "Intel Gatherer" mean the same thing, "Anti-Roam" is sometimes "Anti-Roamer." Let me normalize them, score every operator on a small set of *capabilities*, and cluster the roster into the archetypes pros actually call out in VOD reviews.


In [ ]:
# Normalize role tags into a small set of capability buckets
CAPS = {
    "intel":       ["intel","intel gatherer","intel denier"],
    "anti_gadget": ["anti-gadget","anti gadget","anti-entry","anti entry","utility denier"],
    "breach":      ["breach","breacher","hard breach","secure"],
    "area_denial": ["area denial","crowd control","map control","trapper"],
    "support":     ["support","buff","secure","cover","covering fire","front line"],
    "roam":        ["roam","anti roam","anti-roam","back line"],
    "frag":        ["frontline","front line","disrupt","disrupter"],
}

def capability_vector(role_list):
    flat = " | ".join(r.lower() for r in role_list)
    return {cap: int(any(k in flat for k in kws)) for cap, kws in CAPS.items()}

caps_df = pd.DataFrame([capability_vector(r) for r in ops["roles_list"]])
caps_df.index = ops["name"]

# Plus chassis features
chassis_feat = ops.set_index("name")[["speed","health"]].astype(float)
op_features = caps_df.join(chassis_feat).join(
    ops.set_index("name")[["side"]])
op_features["is_attacker"] = (op_features["side"] == "attacker").astype(int)
op_features = op_features.drop(columns="side")
op_features.head()


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

scaler = StandardScaler()
X = scaler.fit_transform(op_features.values)

# Small K — we want interpretable archetypes, not a confusion of micro-clusters
km = KMeans(n_clusters=6, n_init=10, random_state=42).fit(X)
op_features["cluster"] = km.labels_

# 2D embedding for the operator map
pca = PCA(n_components=2, random_state=42).fit_transform(X)
emb = pd.DataFrame(pca, index=op_features.index, columns=["x","y"])
emb["cluster"] = op_features["cluster"]
emb["side"] = ops.set_index("name").loc[emb.index, "side"]

# Auto-name clusters using their mean capability profile
cap_cols = list(CAPS.keys())
profiles = op_features.groupby("cluster")[cap_cols + ["speed","health","is_attacker"]].mean()

def name_cluster(row):
    top = row[cap_cols].sort_values(ascending=False).head(2).index.tolist()
    side = "ATK" if row["is_attacker"] > 0.5 else "DEF"
    speed_tag = "Roamer" if row["speed"] >= 2.4 else ("Anchor" if row["speed"] <= 1.6 else "Flex")
    cap_tag = " / ".join(t.replace("_"," ").title() for t in top)
    return f"{side} {speed_tag} — {cap_tag}"

cluster_names = {c: name_cluster(profiles.loc[c]) for c in profiles.index}
op_features["archetype"] = op_features["cluster"].map(cluster_names)
emb["archetype"] = emb["cluster"].map(cluster_names)
print("Archetypes discovered:")
for c, n in cluster_names.items():
    print(f"  cluster {c}: {n}  ({(op_features['cluster']==c).sum()} ops)")


In [ ]:
fig = px.scatter(emb.reset_index(), x="x", y="y", color="archetype",
                 symbol="side", hover_name="name",
                 template=PLOTLY_TEMPLATE,
                 color_discrete_sequence=px.colors.qualitative.Bold,
                 title="Operator similarity map — PCA of capabilities + chassis")
fig.update_traces(marker=dict(size=11, line=dict(width=0.6, color="#0b0d12")))
fig.update_layout(height=560, xaxis_title="", yaxis_title="",
                  legend_title="Archetype")
fig.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.heatmap(profiles[cap_cols].round(2), annot=True, cmap="rocket_r",
            linewidths=0.4, linecolor="#0b0d12", cbar_kws={"label":"share of cluster"},
            ax=ax)
ax.set_yticklabels([cluster_names[c] for c in profiles.index], rotation=0)
ax.set_title("What each cluster is actually built to do")
ax.set_xlabel(""); ax.set_ylabel("")
plt.tight_layout(); plt.show()


**Meta read.** The map separates cleanly into two halves (the obvious ATK/DEF split) and then into two flavours within each side: **roamers vs anchors** on defense, **entry vs utility** on attack. Operators sitting on cluster boundaries — Jackal between intel and roam-hunter, Iana between intel and frag — are the ones who *flex roles in scrim*. That floating identity is exactly why they get banned so often.

---
## 6. Meta & Balance Analysis

This is the section I actually care about. Ubisoft's balance problem isn't damage numbers — it's **utility overload**. Some operators carry three or four overlapping tools (gun + gadget + secondary gadget + role utility) and end up dominant on every map. Others have one trick that's strong on two sites and useless on the rest.

Let me try to quantify both.


In [ ]:
# Versatility score: more capabilities + more loadout options = more versatile
cap_count = caps_df.sum(axis=1).rename("capability_count")
primary_count = flex["primary_options"].reindex(ops["name"]).fillna(0).rename("primary_count")
sec_count = (w[~w["type"].isin(primary_classes)][["operators_list"]]
              .explode("operators_list").dropna()
              .rename(columns={"operators_list":"operator"})
              .groupby("operator").size().rename("secondary_count"))

scores = (pd.concat([cap_count, primary_count, sec_count], axis=1)
            .fillna(0).join(ops.set_index("name")[["side","speed","health"]]))
# z-scored versatility score
for c in ["capability_count","primary_count","secondary_count"]:
    scores[c+"_z"] = (scores[c] - scores[c].mean()) / scores[c].std(ddof=0)
scores["versatility"] = scores[["capability_count_z","primary_count_z","secondary_count_z"]].mean(axis=1)

most_versatile  = scores.sort_values("versatility", ascending=False).head(10)
most_specialist = scores.sort_values("versatility").head(10)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(x=most_versatile["versatility"], y=most_versatile.index,
            palette="Oranges_r", ax=axes[0])
axes[0].set_title("Most overloaded operators (utility + loadout breadth)")
axes[0].set_xlabel("Versatility score (z)")
sns.barplot(x=most_specialist["versatility"], y=most_specialist.index,
            palette="Blues", ax=axes[1])
axes[1].set_title("Most specialized operators (one job, do it well)")
axes[1].set_xlabel("Versatility score (z)")
plt.tight_layout(); plt.show()


In [ ]:
# Tradeoff scatter: capability_count vs primary_count, with versatility as bubble size
scores_plot = scores.reset_index().rename(columns={"index":"operator"})
scores_plot["bubble"] = scores_plot["versatility"] - scores_plot["versatility"].min() + 0.4
fig = px.scatter(scores_plot, x="primary_count", y="capability_count",
                 color="side", hover_name="operator", size="bubble",
                 color_discrete_map={"attacker":ATK,"defender":DEF},
                 template=PLOTLY_TEMPLATE, size_max=22,
                 title="Loadout flexibility vs tactical capability count")
fig.update_layout(height=520, xaxis_title="# primary weapons available",
                  yaxis_title="# tactical capabilities (intel, denial, breach, etc.)")
fig.show()


In [ ]:
# Risk/reward: high firepower potential + locked loadout = high-risk pick
op_dps = (op_primary.merge(w[["name","dps_proxy"]].rename(columns={"name":"weapon"}),
                            on="weapon")
                     .groupby("operator")["dps_proxy"].max()
                     .rename("best_dps"))
risk = pd.concat([op_dps, flex["primary_options"]], axis=1).dropna()
risk["risk_reward"] = risk["best_dps"] / risk["primary_options"].clip(lower=1)
high_rr = risk.sort_values("risk_reward", ascending=False).head(10).round(2)
print("High risk / high reward picks (top firepower per loadout option):")
high_rr


**Meta read.** The versatility list reads exactly like a recent ban-phase priority chart — operators with multiple capability tags *and* deep loadouts are the ones that warp drafts. The specialists are honest: one tool, one job, get banned only on specific maps. The bubble chart shows the real balance sweet spot Ubisoft seems to aim for — roughly 2 capabilities and 2 primary options. Anyone living far outside that box is either oppressive in the meta or a niche pick collecting dust.

---
## 7. Operator Recommendation Engine

The "if you like X, try Y" question. Lightweight cosine similarity on the same capability + chassis vectors. No ML overkill — the dataset is small enough that the math itself is the model.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Build the recommendation feature matrix: capabilities + chassis + side
rec_feats = caps_df.join(ops.set_index("name")[["speed","health"]]).copy()
rec_feats["is_attacker"] = (ops.set_index("name")["side"] == "attacker").astype(int)
# Up-weight side (don't recommend defenders to attackers)
rec_feats["is_attacker"] *= 3
rec_mat = StandardScaler().fit_transform(rec_feats.values)
sim = pd.DataFrame(cosine_similarity(rec_mat),
                   index=rec_feats.index, columns=rec_feats.index)

def recommend(op_name, k=5):
    if op_name not in sim.index:
        print(f"'{op_name}' not in roster."); return
    scores = sim[op_name].drop(op_name).sort_values(ascending=False).head(k)
    side = ops.set_index("name").loc[op_name, "side"].title()
    print(f"\nIf you main {op_name} ({side}), try:")
    for name, s in scores.items():
        their_side = ops.set_index("name").loc[name, "side"].title()
        their_roles = ", ".join(ops.set_index("name").loc[name, "roles_list"][:3])
        print(f"  • {name:<14} (sim {s:.2f}, {their_side}) — {their_roles}")

for o in ["Ash", "Smoke", "Thatcher", "Caveira", "Thermite"]:
    recommend(o, k=4)


In [ ]:
# Playstyle presets — pick by capability vector
def by_playstyle(want_caps, side, k=6):
    pool = ops.set_index("name")
    pool = pool[pool["side"] == side]
    feats = caps_df.loc[pool.index]
    # score = how many of the desired capabilities the operator covers
    score = feats[want_caps].sum(axis=1) + 0.1 * pool["speed"]
    return score.sort_values(ascending=False).head(k).round(2)

print("Aggressive entry-frag attackers:")
print(by_playstyle(["frag","breach","area_denial"], "attacker"))
print("\nIntel-leaning support attackers:")
print(by_playstyle(["intel","support","anti_gadget"], "attacker"))
print("\nDefensive utility-deniers:")
print(by_playstyle(["anti_gadget","area_denial"], "defender"))
print("\nRoam-and-hunt defenders:")
print(by_playstyle(["roam","intel"], "defender"))


---
## 8. Competitive Meta Simulation

The real draft question isn't "pick the best operator" — it's "pick five that cover every job a round needs." On attack you need: hard breach, soft destruction, intel, anti-utility, and a frag/entry. On defense: anti-hard-breach, anti-soft-breach, intel denial, anchor, roamer.

I'll do a fast, greedy coverage optimizer. Not Monte Carlo — that's overkill for 78 operators and a 5-slot team.


In [ ]:
ATK_NEEDS = ["intel","anti_gadget","breach","area_denial","support","frag"]
DEF_NEEDS = ["intel","anti_gadget","area_denial","support","roam"]

def greedy_team(side, needs, size=5):
    pool_names = ops[ops["side"] == side]["name"].tolist()
    caps_pool = caps_df.loc[pool_names].copy()
    team, remaining = [], set(needs)
    # Phase 1 — fill the needs
    while remaining and len(team) < size:
        # for each candidate, count how many *remaining* needs they cover
        cover = caps_pool[list(remaining)].sum(axis=1)
        # tiebreak: total capabilities (versatility)
        tie = caps_pool.sum(axis=1)
        ranked = pd.DataFrame({"cover":cover,"tie":tie}).sort_values(["cover","tie"], ascending=False)
        pick = ranked.index[0]
        if ranked.loc[pick,"cover"] == 0: break
        team.append(pick)
        covered = caps_pool.loc[pick]
        for n in list(remaining):
            if covered.get(n, 0) > 0: remaining.discard(n)
        caps_pool = caps_pool.drop(pick)
    # Phase 2 — fill remaining slots with most versatile leftovers
    while len(team) < size and not caps_pool.empty:
        pick = caps_pool.sum(axis=1).sort_values(ascending=False).index[0]
        team.append(pick); caps_pool = caps_pool.drop(pick)
    return team, remaining

atk_team, atk_gap = greedy_team("attacker", ATK_NEEDS)
def_team, def_gap = greedy_team("defender", DEF_NEEDS)

def describe(name):
    row = ops.set_index("name").loc[name]
    return f"{name:<12}  HP{int(row['health'])}/SP{int(row['speed'])}  — {', '.join(row['roles_list'])}"

print("OPTIMAL ATTACK COMP")
for o in atk_team: print("  •", describe(o))
print(f"  uncovered needs: {atk_gap or 'none'}\n")

print("OPTIMAL DEFENSE COMP")
for o in def_team: print("  •", describe(o))
print(f"  uncovered needs: {def_gap or 'none'}")


In [ ]:
# Utility overlap heatmap for the chosen attack team — where's the redundancy?
def overlap_heatmap(team, title):
    mat = caps_df.loc[team]
    fig, ax = plt.subplots(figsize=(8, 3.6))
    sns.heatmap(mat, cmap="rocket_r", cbar=False, linewidths=0.6,
                linecolor="#0b0d12", annot=True, fmt="d", ax=ax)
    ax.set_title(title); ax.set_xlabel(""); ax.set_ylabel("")
    plt.tight_layout(); plt.show()

overlap_heatmap(atk_team, "Attack comp — capability coverage by operator")
overlap_heatmap(def_team, "Defense comp — capability coverage by operator")


**Meta read.** The greedy solver lands on comps that look *familiar* — they're not pixel-perfect copies of pro-league strats, but the *shape* is right: one hard breach, one intel piece, one anti-utility, one anchor/frag, and a flex. Any redundancy in the heatmaps is a hint that two operators on the team are stepping on each other's toes. In live pug Siege, that's where you lose draft fights.

---
## 10. Final Takeaways

A few honest conclusions after sitting with this data:

- **Ubisoft balances the chassis, not the gun.** 2-speed / 2-health is the dead center of the roster, and corner chassis (3/1, 1/3) are doled out carefully — that's the *real* lever they pull.
- **The Nighthaven dump-bin is doing a lot of meta work.** Most of the recent oppressive picks live under the merc faction, not the legacy CT units.
- **Loadout flexibility is the cleanest balance signal.** Operators with only one primary class genuinely *do* require team accommodation, and that opportunity cost shows up in pick rates.
- **The role tags get the archetypes mostly right** — but the boundary operators (Iana, Jackal, Brava, Solis) are exactly the ones that warp drafts because they refuse to sit in a single cluster.
- **A small, honest recommender beats a fancy one here.** With 78 operators, cosine similarity on a normalized capability vector is enough. ML wouldn't add anything except runtime.

### What this notebook *can't* see
- **Live pick/ban data**, which is the closest thing to ground truth on the meta. Without it, "balance" is a structural argument, not an empirical one.
- **Gadget interactions** (Mute jammer vs Thatcher EMP vs Kali LV, etc.) — these aren't in the encyclopedia at the level of detail that would let us model counter-picks properly.
- **Map-specific viability.** A defender who's god-tier on Clubhouse is a coin-flip on Coastline. Without per-map stats it's a roster-wide average.
- **Patch context.** A snapshot dataset can't track the constant rebalancing that defines Siege's identity.

### What I'd add next
1. Pro-league pick/ban scrape per patch — that single dataset turns half of this into a real predictive model.
2. Gadget interaction graph (who counters whom).
3. Site-by-site viability scores.
4. ELO of each operator over patch history.

That's the whole sport of Siege analytics: the structure is in the data, the meta is in the patches, and the truth is in the scrim VODs. This notebook is the first of those three.
